In [3]:
import pandas as pd
import numpy as np

from google.colab import files

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [5]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]

# Read Excel file
df = pd.read_excel(filename)

print("Dataset Shape:", df.shape)
df.head()

Saving financial_inclusion_dataset (1).xlsx to financial_inclusion_dataset (1).xlsx
Dataset Shape: (30, 9)


,Age,Gender,Education,Employment,Annual_Income,Mobile_Phone,Internet_Access,Loan_History,Bank_Account
0,22,Male,Graduate,Employed,25000,Yes,Yes,Good,Yes
1,35,Female,Postgraduate,Self-Employed,45000,Yes,Yes,Good,Yes
2,28,Male,Higher Secondary,Unemployed,12000,Yes,No,Poor,No
3,45,Female,Graduate,Government,60000,Yes,Yes,Good,Yes
4,31,Male,Graduate,Private,38000,Yes,Yes,Average,Yes


In [6]:
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print(df.isnull().sum())

Age                0
Gender             0
Education          0
Employment         0
Annual_Income      0
Mobile_Phone       0
Internet_Access    0
Loan_History       0
Bank_Account       0
dtype: int64


In [7]:
encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = encoder.fit_transform(df[col])

df.head()

,Age,Gender,Education,Employment,Annual_Income,Mobile_Phone,Internet_Access,Loan_History,Bank_Account
0,22,1,0,1,25000,1,1,1,1
1,35,0,2,4,45000,1,1,1,1
2,28,1,1,6,12000,1,0,2,0
3,45,0,0,2,60000,1,1,1,1
4,31,1,0,3,38000,1,1,0,1


In [8]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nClass Distribution")
print(y.value_counts())

Features Shape: (30, 8)
Target Shape: (30,)

Class Distribution
Bank_Account
1    21
0     9
Name: count, dtype: int64


In [9]:
minimum = y.value_counts().min()

if minimum >= 10:
    folds = 10
elif minimum >= 5:
    folds = 5
elif minimum >= 3:
    folds = 3
else:
    folds = 2

print("Using", folds, "Fold Stratified K-Fold")

skf = StratifiedKFold(
    n_splits=folds,
    shuffle=True,
    random_state=42
)

Using 5 Fold Stratified K-Fold


In [10]:
model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

In [11]:
accuracy = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring='accuracy'
)

print("Accuracy of Each Fold")
print(np.round(accuracy*100,2))

print("\nAverage Accuracy")
print(round(accuracy.mean()*100,2),"%")

Accuracy of Each Fold
[100.   100.    83.33 100.   100.  ]

Average Accuracy
96.67 %


In [12]:
precision = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring='precision_weighted'
)

print("Average Precision")
print(round(precision.mean()*100,2),"%")

Average Precision
97.33 %


In [13]:
recall = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring='recall_weighted'
)

print("Average Recall")
print(round(recall.mean()*100,2),"%")

Average Recall
96.67 %


In [14]:
f1 = cross_val_score(
    model,
    X,
    y,
    cv=skf,
    scoring='f1_weighted'
)

print("Average F1 Score")
print(round(f1.mean()*100,2),"%")

Average F1 Score
96.3 %


In [15]:
print("===================================")
print("Machine Learning Performance")
print("===================================")

print("Accuracy :", round(accuracy.mean()*100,2), "%")
print("Precision:", round(precision.mean()*100,2), "%")
print("Recall   :", round(recall.mean()*100,2), "%")
print("F1 Score :", round(f1.mean()*100,2), "%")

Machine Learning Performance
Accuracy : 96.67 %
Precision: 97.33 %
Recall   : 96.67 %
F1 Score : 96.3 %
